In [1]:
from langchain_classic import LlamaCpp

In [2]:
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [3]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

We got no output as Phi-3 requires a specific prompt template.

Lets create a prompt template adhering Phi-3's expectation, and utilize the llm model to create our first chain. 

In [4]:
from langchain_classic import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [5]:
basic_chain = prompt | llm

In [6]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

Now the chain is set up to take an input prompt and pass it to the LLM for processing. 
We don't need to worry about the formatting of the prompt every time we use LLM.

Multiple Chains

In [7]:
from langchain_classic import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

C:\Users\rosha\AppData\Local\Temp\ipykernel_16592\1767282371.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [8]:
title.invoke({"summary": "a girl that lost her mother"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Farewell: The Journey Through Loss"'}

In [9]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [10]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [11]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [12]:
llm_chain.invoke("a musician who has become war victim but still chose to spread love with his music after the war was over.")

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a musician who has become war victim but still chose to spread love with his music after the war was over.',
 'title': ' "Melodies of Resilience: A War Survivor\'s Journey to Heal with Harmonious Love"',
 'character': ' "Melodies of Resilience" tells the poignant tale of a passionate musician who, after enduring unimaginable wartime horrors, chooses to channel his pain into creating soul-stirring melodies that embody hope and unity. His hauntingly beautiful compositions become beacons of love and resilience in post-war society, inspiring healing through the universal language of music.',
 'story': ' In the heart-wrenching narrative "Melodies of Resilience: A War Survivor\'s Journey to Heal with Harmonious Love," we delve into the poignant life of an unnamed musician whose world was shattered by war. Amidst the ruins, our protagonist discovers solace in his violin strings, each note a testament to survival and an emblem of healing. The hauntingly beautiful melodies that eme

Memory

In [13]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

In [14]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" I'm unable to determine your name as I don't have access to personal data of individuals. If you need assistance with something specific, feel free to ask!"

Stateless LLMs have no memory of previous conversation.

Conversation Buffer

In [15]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [16]:
from langchain_classic.memory import ConversationBufferMemory

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

C:\Users\rosha\AppData\Local\Temp\ipykernel_16592\3768838688.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history")


In [17]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': " The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units in total.\n\nHere is the calculation:\n\n1 + 1 = 2"}

In [18]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units in total.\n\nHere is the calculation:\n\n1 + 1 = 2",
 'text': " Your name is Maarten.\n\nI'm an AI and I don't have a personal name, but you can call me Assistant."}

ConversationBufferMemoryWindow

In [19]:
from langchain_classic.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

C:\Users\rosha\AppData\Local\Temp\ipykernel_16592\1769901941.py:4: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")


In [20]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! It's nice to meet you. 1 + 1 equals 2.",
 'text': ' Hello! 3 + 3 equals 6.'}

In [21]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! It's nice to meet you. 1 + 1 equals 2.\nHuman: What is 3 + 3?\nAI:  Hello! 3 + 3 equals 6.",
 'text': ' Your name is Maarten.'}

In [22]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': 'Human: What is 3 + 3?\nAI:  Hello! 3 + 3 equals 6.\nHuman: What is my name?\nAI:  Your name is Maarten.',
 'text': " I do not have the ability to know personal information about individuals unless it has been previously shared with me in our conversation. Therefore, I cannot tell you your name without that context. Please provide the name if you'd like assistance related to it."}

Here, chat history is limited to the last two conversations.
The LLM indeed has no access to our age since that was not retained in the chat history.

Conversation Summary Memory

In [23]:
# Create a summary prompt template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [24]:
from langchain_classic.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

C:\Users\rosha\AppData\Local\Temp\ipykernel_16592\1883484148.py:4: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(


In [25]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': ' Maarten introduces himself and asks the AI a simple arithmetic question about addition, to which the AI correctly responds that 1 + 1 equals 2. The explanation also includes an illustrative example of how the operation is performed.',
 'text': " I don't have the ability to know personal information about individuals unless it has been shared in our conversation. My name is Microsoft's Assistant.\n\nAs for your arithmetic question, if you are adding 1 + 1, the result indeed equals 2. Here's a simple illustrative example: Imagine you have one apple and then receive another apple, making two apples altogether. Mathematically, that translates to the equation 1 + 1 = 2."}

In [26]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What was the first question I asked?',
 'chat_history': " Maarten initiates the conversation by introducing himself and posing a basic arithmetic question about addition to Microsoft's Assistant, which correctly answers that 1 + 1 equals 2. The AI provides an illustrative example of adding apples to explain the operation further. When asked for personal information, such as their name, the AI clarifies its limitations in accessing private data but identifies itself as Microsoft's Assistant.",
 'text': ' The first question you asked was, "1 + 1 equals what?"'}

In [27]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': ' Maarten starts the conversation by introducing himself and asks Microsoft\'s Assistant a basic arithmetic question: "1 + 1 equals what?" The AI correctly answers that it is equal to 2, providing an example of adding apples. When asked for personal information like their name, the AI reminds its limitations in accessing private data but identifies itself as Microsoft\'s Assistant. Additionally, when queried about the first question posed during the conversation, the AI responds that it was "1 + 1 equals what?"'}

Agents

In [51]:
import os
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAI

In [ ]:
# Load OpenAI's LLMs with LangChain
os.environ["OPENAI_API_KEY"] = "YOUR KEY HERE"
openai_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0) 

In [90]:
# Create the ReAct template
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

In [91]:
from langchain_community.agent_toolkits.load_tools import load_tools, Tool
from  langchain_community.tools import DuckDuckGoSearchResults

# You can create the tool to pass to an agent
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)

# Prepare tools
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

In [92]:
from langchain_classic.agents import AgentExecutor, create_react_agent

# Construct the ReAct agent
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [94]:
# What is the Price of a MacBook Pro?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
    }
)



> Entering new AgentExecutor chain...


KeyboardInterrupt: 